## Starting Mini 01

In [2]:
import pandas as pd
import os
import pymysql
import json

In [3]:
print("Here is my working directory:", os.getcwd())

Here is my working directory: /Users/andrewhimmelman/Documents/AndrewHimmelman/BAS 479/Case 01


## Initial Things

In [9]:
## Let's get the SQL data inside

with open("credentials.json") as f:
    creds = json.load(f)

con = pymysql.connect(
    host=creds["host"],
    user=creds["username"],
    password=creds["password"],
    database="aspannba_bas479"
)

carbo_transactions = pd.read_sql(
    "SELECT * FROM carbo_transactions;",
    con
)

print(carbo_transactions)

# Comment out the close for now but we will close later after we are done

# con.close()


/var/folders/vk/lswj183x58bbxh5t1j_7qqpc0000gn/T/ipykernel_10967/2266598589.py:15: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  carbo_transactions = pd.read_sql(


               upc  dollar_sales  units  time_of_transaction  geography  week  \
0       9999985031          1.58      2                    9          1    43   
1       9999985058          1.58      2                    9          1    43   
2       3620000445          3.22      2                  106          1    43   
3       9999985006          1.39      1                  106          1    43   
4       2920000214          1.09      1                   51          1    43   
...            ...           ...    ...                  ...        ...   ...   
324355  4420979347          4.49      1                 1859          2    72   
324356  7107201136          3.19      1                 1859          2    72   
324357  9999981816          4.29      1                 2321          1    72   
324358  3000005070          1.99      1                 2312          1    72   
324359  7130000027          1.98      2                 2339          1    72   

       household  store   b

In [5]:

## Also we need to make sure that we load in the other files

carbo_product_lookup = pd.read_csv("Data/carbo_product_lookup.csv")

carbo_promotion_lookup = pd.read_json("Data/carbo_promotion_lookup.json")

print("Everything seems to be loaded in correctly.",
      "Carbo_Product Shape:", carbo_product_lookup.shape,
      "Carbo_Promotion Shape:", carbo_promotion_lookup.shape)

Everything seems to be loaded in correctly. Carbo_Product Shape: (670, 5) Carbo_Promotion Shape: (17263, 6)


## EDA, EDA, EDA!

In [10]:
# Kahoootinnn

q1 = carbo_transactions.shape
print(q1)

(324360, 11)


In [13]:
# What percent of records have a coupon applied?

q2 = pd.read_sql(
    "SELECT COUNT(*) FROM carbo_transactions WHERE coupon != 0", 
    con
)
print(q2/len(carbo_transactions))

   COUNT(*)
0  0.016747


/var/folders/vk/lswj183x58bbxh5t1j_7qqpc0000gn/T/ipykernel_10967/209547148.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  q2 = pd.read_sql(


In [14]:
q3 = pd.read_sql(
    "SELECT COUNT(*) FROM carbo_transactions WHERE coupon != 0 AND store = 281", 
    con)

print(q3/len(carbo_transactions[carbo_transactions["store"] == 281]))

   COUNT(*)
0  0.010224


/var/folders/vk/lswj183x58bbxh5t1j_7qqpc0000gn/T/ipykernel_10967/2217263239.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  q3 = pd.read_sql("SELECT COUNT(*) FROM carbo_transactions WHERE coupon != 0 AND store = 281", con)


In [17]:
# simple group by

q4 = pd.read_sql(
    """
    SELECT upc, SUM(units) AS transaction_count
    FROM carbo_transactions
    GROUP BY upc
    ORDER BY transaction_count DESC
    """,
    con
)

print(q4)

/var/folders/vk/lswj183x58bbxh5t1j_7qqpc0000gn/T/ipykernel_10967/3041308049.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  q4 = pd.read_sql(


            upc  transaction_count
0    9999985020            10184.0
1    9999985004             8575.0
2    3620000250             6985.0
3    9999985068             6850.0
4    3620000300             4488.0
..          ...                ...
683  5346901004                1.0
684  7680851398                1.0
685  4747900037                1.0
686  5346901001                1.0
687  7546203832                1.0

[688 rows x 2 columns]


In [ ]:
# i got wrong

q5 = pd.read_sql(
    """
    SELECT store, basket, avg(units_per_basket) as avg_units_per_basket
    FROM (
        SELECT store, basket, sum(units) as units_per_basket
        FROM carbo_transactions
    ) AS subquery
    GROUP BY store, basket
    ORDER BY units_per_basket DESC
    """,
    con)


print(q5)

/var/folders/vk/lswj183x58bbxh5t1j_7qqpc0000gn/T/ipykernel_10967/2440873242.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  q5 = pd.read_sql(


   store   basket  avg_units_per_basket
0     16  1304172              386203.0
